# Embed Reddit Titles With DistilBERT

## Assumptions
- Input parquet already exists at `data/processed/title_embedding_input.parquet`.
- Each post is represented by the DistilBERT embedding of `title_for_embedding`.
- Title embeddings use masked mean pooling over `last_hidden_state` because DistilBERT does not expose a pooler head.
- Each subreddit is represented by the arithmetic mean of its post embeddings.
- Subreddit relationships are cosine similarities between subreddit vectors.

## Outputs
- `data/processed/subreddit_title_embeddings_distilbert.parquet`
- `data/processed/subreddit_cosine_similarity_distilbert.parquet`

In [14]:
# Import cac thu vien can cho embedding va cosine similarity.
from pathlib import Path
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModel, AutoTokenizer, logging as transformers_logging

# Giam bot warning khi nap model tu Hugging Face.
transformers_logging.set_verbosity_error()

# Cho phep chay notebook tu repo root hoac tu thu muc notebooks.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent

# Khai bao duong dan input va output cua pipeline embedding.
INPUT_PATH = Path("/kaggle/input/datasets/nhquan0811/subreddit/title_embedding_input.parquet")
OUTPUT_DIR = Path("/kaggle/working")
SUBREDDIT_EMBEDDINGS_PATH = OUTPUT_DIR / "subreddit_title_embeddings_distilbert.parquet"
SIMILARITY_PATH = OUTPUT_DIR / "subreddit_cosine_similarity_distilbert.parquet"

# Cau hinh model, batch size va thiet bi chay.
MODEL_NAME = "distilbert-base-uncased"
MODEL_BATCH_SIZE = 64
PARQUET_BATCH_SIZE = 2048
MAX_LENGTH = 64
PREVIEW_ROWS = 5
SIMILARITY_EDGE_PERCENTILE = 97.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INPUT_PATH

PosixPath('/kaggle/input/datasets/nhquan0811/subreddit/title_embedding_input.parquet')

In [15]:
# Mean pooling co mask de bien chuoi token thanh 1 vector title duy nhat.
def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_sum = (last_hidden_state * mask).sum(dim=1)
    token_count = mask.sum(dim=1).clamp(min=1.0)
    return masked_sum / token_count


# Tokenize va lay embedding cho mot batch title.
def encode_titles(titles: list[str], tokenizer: AutoTokenizer, model: AutoModel) -> np.ndarray:
    encoded = tokenizer(
        titles,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded = {key: value.to(DEVICE) for key, value in encoded.items()}

    with torch.inference_mode():
        outputs = model(**encoded)

    embeddings = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
    return embeddings.cpu().numpy().astype(np.float32)


# Duyet file parquet theo batch, tinh embedding title va cong don theo subreddit.
def build_subreddit_embeddings(input_path: Path, tokenizer: AutoTokenizer, model: AutoModel):
    parquet_file = pq.ParquetFile(input_path)
    subreddit_sums: dict[str, np.ndarray] = {}
    subreddit_counts: dict[str, int] = {}
    preview_rows: list[dict[str, object]] = []
    processed_rows = 0
    next_progress_mark = 100_000

    for parquet_batch in parquet_file.iter_batches(
        batch_size=PARQUET_BATCH_SIZE,
        columns=["name", "subreddit", "title_for_embedding"],
    ):
        frame = parquet_batch.to_pandas().dropna(subset=["subreddit", "title_for_embedding"]).reset_index(drop=True)
        if frame.empty:
            continue

        titles = frame["title_for_embedding"].tolist()
        batch_vectors: list[np.ndarray] = []

        # Chia nho tiep thanh cac batch model de tranh vuot bo nho.
        for start in range(0, len(titles), MODEL_BATCH_SIZE):
            stop = start + MODEL_BATCH_SIZE
            batch_vectors.append(encode_titles(titles[start:stop], tokenizer, model))

        embeddings = np.vstack(batch_vectors)
        processed_rows += len(frame)

        # Luu mot vai post dau de xem nhanh embedding mau trong notebook.
        preview_slots = PREVIEW_ROWS - len(preview_rows)
        if preview_slots > 0:
            preview_frame = frame.head(preview_slots).copy()
            preview_frame["post_embedding"] = embeddings[:preview_slots].tolist()
            preview_rows.extend(preview_frame.to_dict("records"))

        # Moi subreddit duoc bieu dien boi tong vector va so post de cuoi cung lay trung binh.
        for subreddit, embedding in zip(frame["subreddit"].tolist(), embeddings):
            if subreddit in subreddit_sums:
                subreddit_sums[subreddit] += embedding
                subreddit_counts[subreddit] += 1
            else:
                subreddit_sums[subreddit] = embedding.astype(np.float64)
                subreddit_counts[subreddit] = 1

        # In tien do de de theo doi khi chay tren tap du lieu lon.
        while processed_rows >= next_progress_mark:
            print(f"Processed {next_progress_mark:,} titles.")
            next_progress_mark += 100_000

    if not subreddit_sums:
        raise ValueError(f"No titles were loaded from {input_path}.")

    subreddits = sorted(subreddit_sums)

    # Chuyen tong vector thanh mean vector cua tung subreddit.
    embedding_matrix = np.vstack(
        [
            (subreddit_sums[subreddit] / subreddit_counts[subreddit]).astype(np.float32)
            for subreddit in subreddits
        ]
    )
    subreddit_embeddings = pd.DataFrame(
        {
            "subreddit": subreddits,
            "post_count": [subreddit_counts[subreddit] for subreddit in subreddits],
            "embedding": embedding_matrix.tolist(),
        }
    )
    post_embedding_preview = pd.DataFrame(preview_rows)

    return subreddit_embeddings, embedding_matrix, post_embedding_preview, processed_rows


# Tinh cosine similarity giua cac vector subreddit.
def build_similarity_frame(subreddit_embeddings: pd.DataFrame, embedding_matrix: np.ndarray) -> pd.DataFrame:
    similarity_matrix = cosine_similarity(embedding_matrix).astype(np.float32)
    similarity_frame = pd.DataFrame(
        similarity_matrix,
        index=subreddit_embeddings["subreddit"],
        columns=subreddit_embeddings["subreddit"],
    )
    return similarity_frame.reset_index(names="subreddit")


# Chuyen ma tran similarity thanh edge list va giu lai nhom similarity cao nhat.
def build_similarity_edges(similarity_frame: pd.DataFrame, percentile: float = 97.0) -> tuple[pd.DataFrame, float]:
    if not 0 <= percentile <= 100:
        raise ValueError("percentile must be between 0 and 100.")

    similarity_values = similarity_frame.iloc[:, 1:].to_numpy(dtype=np.float32, copy=False)
    subreddits = similarity_frame["subreddit"].tolist()
    row_idx, col_idx = np.triu_indices(len(subreddits), k=1)
    edge_frame = pd.DataFrame(
        {
            "source_subreddit": [subreddits[idx] for idx in row_idx],
            "target_subreddit": [subreddits[idx] for idx in col_idx],
            "cosine_similarity": similarity_values[row_idx, col_idx],
        }
    )
    similarity_threshold = float(
        np.percentile(edge_frame["cosine_similarity"].to_numpy(dtype=np.float32, copy=False), percentile)
    )
    filtered_edges = (
        edge_frame.loc[edge_frame["cosine_similarity"] >= similarity_threshold]
        .sort_values(["cosine_similarity", "source_subreddit", "target_subreddit"], ascending=[False, True, True])
        .reset_index(drop=True)
    )
    return filtered_edges, similarity_threshold

In [16]:
# Kiem tra dau vao truoc khi chay pipeline embedding.
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Missing input parquet: {INPUT_PATH}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
input_file = pq.ParquetFile(INPUT_PATH)
expected_rows = input_file.metadata.num_rows

pipeline_start_time = time.perf_counter()

# Nap tokenizer va DistilBERT model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# Tao vector dai dien cho tung subreddit tu title embeddings.
embedding_start_time = time.perf_counter()
subreddit_embeddings, embedding_matrix, post_embedding_preview, processed_rows = build_subreddit_embeddings(
    INPUT_PATH,
    tokenizer,
    model,
)
embedding_elapsed_seconds = round(time.perf_counter() - embedding_start_time, 2)
print(f"Embedding completed in {embedding_elapsed_seconds} seconds.")

# Tinh cosine similarity giua cac subreddit.
similarity_start_time = time.perf_counter()
similarity_frame = build_similarity_frame(subreddit_embeddings, embedding_matrix)
similarity_elapsed_seconds = round(time.perf_counter() - similarity_start_time, 2)
print(f"Cosine similarity completed in {similarity_elapsed_seconds} seconds.")

# Loc edge list theo percentile de chi giu cac cap subreddit giong nhau nhat.
strong_similarity_edges, similarity_edge_threshold = build_similarity_edges(
    similarity_frame,
    percentile=SIMILARITY_EDGE_PERCENTILE,
)

# Luu output de dung lai o cac buoc phan tich sau.
subreddit_embeddings.to_parquet(SUBREDDIT_EMBEDDINGS_PATH, index=False)
strong_similarity_edges.to_parquet(SIMILARITY_PATH, index=False)

# Tong hop thong tin chay de xem nhanh trong notebook.
summary = {
    "device": str(DEVICE),
    "model_name": MODEL_NAME,
    "processed_titles": processed_rows,
    "expected_titles": expected_rows,
    "subreddits": len(subreddit_embeddings),
    "embedding_size": int(embedding_matrix.shape[1]),
    "embedding_elapsed_seconds": embedding_elapsed_seconds,
    "similarity_elapsed_seconds": similarity_elapsed_seconds,
    "similarity_edge_percentile": SIMILARITY_EDGE_PERCENTILE,
    "similarity_edge_threshold": similarity_edge_threshold,
    "strong_edge_count": len(strong_similarity_edges),
    "subreddit_embeddings_path": str(SUBREDDIT_EMBEDDINGS_PATH),
    "similarity_path": str(SIMILARITY_PATH),
    "elapsed_seconds": round(time.perf_counter() - pipeline_start_time, 2),
}

summary

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Processed 100,000 titles.
Processed 200,000 titles.
Processed 300,000 titles.
Processed 400,000 titles.
Processed 500,000 titles.
Processed 600,000 titles.
Processed 700,000 titles.
Processed 800,000 titles.
Processed 900,000 titles.
Processed 1,000,000 titles.
Processed 1,100,000 titles.
Processed 1,200,000 titles.
Processed 1,300,000 titles.
Processed 1,400,000 titles.
Processed 1,500,000 titles.
Processed 1,600,000 titles.
Processed 1,700,000 titles.
Processed 1,800,000 titles.
Processed 1,900,000 titles.
Processed 2,000,000 titles.
Processed 2,100,000 titles.
Processed 2,200,000 titles.
Processed 2,300,000 titles.
Embedding completed in 2910.18 seconds.
Cosine similarity completed in 0.13 seconds.


{'device': 'cuda',
 'model_name': 'distilbert-base-uncased',
 'processed_titles': 2356505,
 'expected_titles': 2356505,
 'subreddits': 2999,
 'embedding_size': 768,
 'embedding_elapsed_seconds': 2910.18,
 'similarity_elapsed_seconds': 0.13,
 'similarity_edge_percentile': 97.0,
 'similarity_edge_threshold': 0.9816910028457642,
 'strong_edge_count': 134866,
 'subreddit_embeddings_path': '/kaggle/working/subreddit_title_embeddings_distilbert.parquet',
 'similarity_path': '/kaggle/working/subreddit_cosine_similarity_distilbert.parquet',
 'elapsed_seconds': 2918.59}

In [17]:
# Kiem tra so luong ban ghi sau khi xu ly co khop voi input khong.
assert processed_rows == expected_rows
assert subreddit_embeddings["post_count"].sum() == expected_rows
assert similarity_frame.shape == (len(subreddit_embeddings), len(subreddit_embeddings) + 1)

# Duong cheo cosine similarity phai xap xi 1 va khong duoc co gia tri loi.
similarity_values = similarity_frame.iloc[:, 1:].to_numpy(dtype=np.float32, copy=False)
assert np.allclose(np.diag(similarity_values), 1.0, atol=1e-5)
assert np.isfinite(similarity_values).all()

# Cac edge sau khi loc percentile phai nam tren nguong da tinh.
assert len(strong_similarity_edges) > 0
assert (strong_similarity_edges["cosine_similarity"] >= similarity_edge_threshold).all()

print("Notebook outputs look consistent.")

# Xem thu mot vai post dau cung embedding cua title.
post_embedding_preview.head(PREVIEW_ROWS)

Notebook outputs look consistent.


,name,subreddit,title_for_embedding,post_embedding
0,t3_p7gnjm,0sanitymemes,time to drink up,"[0.3243521451950073, 0.07920306921005249, 0.25..."
1,t3_phtnzk,0sanitymemes,There is always another way,"[0.02178495191037655, -0.13360163569450378, 0...."
2,t3_qifjzf,0sanitymemes,Don’t mean to get political but,"[0.24530482292175293, -0.0810445100069046, 0.0..."
3,t3_p2dzlj,0sanitymemes,My life was better before I knew they existed,"[0.14674200117588043, 0.0804266706109047, -0.0..."
4,t3_qhmbtp,0sanitymemes,What Breed is Kal'tsit,"[0.2309621274471283, 0.13126936554908752, -0.1..."


In [18]:
# Chon thu mot subreddit va xem 10 subreddit gan nhat theo cosine similarity.
target_subreddit = subreddit_embeddings.iloc[0]["subreddit"]

top_neighbors = (
    similarity_frame.set_index("subreddit")
    .loc[target_subreddit]
    .drop(labels=target_subreddit)
    .sort_values(ascending=False)
    .head(10)
    .rename_axis("neighbor_subreddit")
    .reset_index(name="cosine_similarity")
)

top_neighbors

,neighbor_subreddit,cosine_similarity
0,ApexOutlands,0.997169
1,AnimeOT,0.996818
2,AvatarMemes,0.996772
3,BokuNoMetaAcademia,0.996641
4,BlueGhost,0.996219
5,Blacksouls2,0.995715
6,ArcaneOdyssey,0.995418
7,196x,0.995270
8,BlackMetalMemes,0.995171
9,CoachCorySubmissions,0.995121


In [19]:
from pathlib import Path

for p in Path("/kaggle/input/datasets/nhquan0811/subreddit").iterdir():
    print(p)

/kaggle/input/datasets/nhquan0811/subreddit/title_embedding_input.parquet
